In [ ]:
import folium
import joblib
import numpy as np
from dask.distributed import Client as DaskClient
from matplotlib import colors

%matplotlib inline


from utils import (
    BBOX,
    BBOX_GDF,
    DATETIME,
    MODE,
    classes,
    load_data,
    load_embeddings,
    predict,
)

In [ ]:
%reload_ext autoreload
%autoreload 2

## Find and load data

Load data and set up your array to use for prediction

In [ ]:
BBOX_GDF.explore()

In [ ]:
if MODE == "traditional":
    merged = load_data(BBOX, DATETIME)
elif MODE == "embeddings":
    merged = load_embeddings(BBOX, DATETIME)

In [ ]:
merged

In [ ]:
loaded_model = joblib.load(f"test_model_{MODE}.dump")

In [ ]:
loaded_data = merged.compute()

## Run ML classifier

In [ ]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit="16GB")

In [ ]:
predicted = predict(loaded_model, loaded_data)

In [ ]:
predicted

In [ ]:
# Write GeoTIFF
predicted.lulc.odc.write_cog(f"lulc_{DATETIME}_{MODE}.tif", overwrite=True)

In [ ]:
# Lookup table: row = class code -> RGBA; 0 (nodata) stays transparent
lut = np.zeros((max(code for code, *_ in classes) + 1, 4), "uint8")
for code, _, color in classes:
    lut[code] = np.array(colors.to_rgba(color)) * 255

m = predicted.lulc.odc.explore(cmap=lut, name="lulc")

# Legend: "code: label" with class colour
legend = "".join(
    f'<div><span style="background:{color};display:inline-block;width:12px;height:12px;margin-right:6px"></span>{code}: {name}</div>'
    for code, name, color in classes
)
m.get_root().html.add_child(
    folium.Element(
        f'<div style="position:fixed;bottom:30px;right:10px;z-index:1000;background:white;padding:6px 8px;border-radius:4px;font:12px sans-serif">{legend}</div>'
    )
)
m